# 技能4 · Day 5 上机：商业模式画布 + 投资评估

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 核心任务

为 AI 营销 Agent SaaS「MarketingAgent Pro」构建完整投资评估：
1. 商业模式画布（9宫格结构化）
2. DCF 估值（NPV / IRR / 回收期 / PI）
3. 蒙特卡洛模拟（估值分布 + 概率分析）
4. 敏感性分析（龙卷风图）
5. 天道推演多路径场景分析

**真实库**：numpy-financial（NPV/IRR）｜ scipy.stats（蒙特卡洛）｜ pandas + matplotlib
**真实数据**：HubSpot 2023 财报 + Jasper AI Crunchbase + 独立教材 MarketingAgent Pro 单位经济模型


## 0. 环境准备

首次运行需安装依赖（取消注释执行一次）：

> 需要 numpy-financial, scipy, pandas, matplotlib。通常已随 conda/venv 安装。
> numpy-financial 提供 NPV/IRR 等标准金融函数。


In [ ]:
# !pip install numpy-financial scipy pandas matplotlib -q
import numpy as np
import pandas as pd
import numpy_financial as npf
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("环境就绪: numpy-financial + scipy.stats + pandas + matplotlib")


## 1. 商业模式画布（Business Model Canvas）

AI 商业模式画布在传统九宫格基础上适配 AI 原生特征：
- **收入流**：新增 outcome-based pricing + Agent 交易费
- **核心资源**：新增数据资产 + AI 模型 + 算力
- **核心活动**：新增模型训练/评估 + Agent 运维
- **成本结构**：新增推理成本（持续运营成本）

**案例**：MarketingAgent Pro - AI 原生营销 Agent 平台
- 数据校准：HubSpot 2023 财报（gross margin ~78%）、Jasper AI（$125M ARR）、独立教材 Day 5 单位经济模型


In [ ]:
# TODO 1：商业模式画布数据结构
# 提示：用 pandas DataFrame 构建9宫格画布
#   列: '构件', 'MarketingAgent Pro', '传统SaaS对比'
#   9个构件: 客户细分/价值主张/渠道/客户关系/收入流/核心资源/核心活动/核心伙伴/成本结构
#   MarketingAgent Pro 数据参考独立教材Day5综合案例
#   传统SaaS对比列展示AI适配的变化
# 要求：构建DataFrame并打印

# ===== 你的代码 =====
canvas_df = None  # 构建9宫格画布DataFrame

raise NotImplementedError


## 2. DCF 估值模型

DCF（Discounted Cash Flow）是投资评估的核心方法。对 AI SaaS：

| 参数 | 值 | 来源 |
|------|-----|------|
| 初始投资 | $2,000K | 开发团队 + GTM 投入 |
| ARPU | $24K/年 ($2K/月) | 独立教材 Day 5 |
| 毛利率 | 65% | 含推理成本 30% + 数据 5% |
| 折现率 | 15% | VC 典型 SaaS 要求回报 |
| 评估窗口 | 5年 | J 曲线效应需 3-5 年 |

**numpy-financial 核心函数**：
- `npf.npv(rate, cashflows)` — 净现值
- `npf.irr(cashflows)` — 内部收益率


In [ ]:
# TODO 2：DCF 5年财务模型 + NPV 计算
# 提示：1) initial_investment = 2000 ($K)
#       2) customers = [0, 30, 80, 160, 260, 380]
#       3) arpu_annual = 24 ($K/年), gross_margin = 0.65
#       4) opex = [0, 800, 1200, 1800, 2500, 3200]
#       5) fcf = gross_profit - opex, fcf[0] = -initial_investment
#       6) npv = npf.npv(0.15, fcf)
# 要求：构建DCF表格(DataFrame)并计算NPV

# ===== 你的代码 =====
dcf_df = None    # DCF财务模型DataFrame
npv = None       # NPV值

raise NotImplementedError


In [ ]:
# TODO 3：IRR + 回收期 + 盈利指数
# 提示：1) irr = npf.irr(fcf)
#       2) 回收期: 累计现金流首次转正的时点(手动计算)
#          cumulative += cf, 当 cumulative >= 0 时:
#          payback = (i-1) + (-prev_cum) / cf
#       3) PI = PV(未来现金流) / |初始投资|
#          pv_future = sum(fcf[i] / (1+dr)**i for i in range(1, len(fcf)))
# 要求：计算三项投资指标并判断可行性

# ===== 你的代码 =====
irr = None              # IRR
payback_period = None   # 回收期(年)
pi = None               # 盈利指数

raise NotImplementedError


## 3. 蒙特卡洛模拟（Monte Carlo Simulation）

DCF 给出 NPV 的点估计，但 AI SaaS 的关键参数高度不确定：
- **推理成本**：模型 API 价格快速变化（GPT-4 -> DeepSeek 成本降 90%+）
- **ARPU**：outcome-based pricing 下波动大
- **客户增长**：市场竞争 + 产品成熟度不确定
- **毛利率**：推理成本曲线决定长期毛利

蒙特卡洛方法：对不确定参数抽样 -> 计算每次抽样的 NPV -> 得到估值分布

**scipy.stats / numpy 分布**：
- `np.random.normal(mu, sigma, n)` — 正态分布抽样
- `np.clip(arr, low, high)` — 截断分布范围
- `np.percentile(arr, q)` — 分位数


In [ ]:
# TODO 4：蒙特卡洛模拟（10000次）
# 提示：1) 定义参数分布:
#    - ARPU ~ Normal(24, 3)  $K/年
#    - Gross margin ~ Normal(0.65, 0.05), clip to [0.35, 0.85]
#    - Growth multiplier ~ Normal(1.0, 0.2), clip to [0.5, 1.5]
#    - OpEx multiplier ~ Normal(1.0, 0.15)
#       2) 每次抽样: 缩放base case客户数/收入/成本, 计算NPV
#       3) 统计: 均值/中位数/P5/P95/P(NPV>0)
# 要求：运行10000次模拟, 打印估值分布统计量

# ===== 你的代码 =====
npv_sim = None  # 10000次模拟的NPV数组

raise NotImplementedError


## 4. 敏感性分析（龙卷风图）

龙卷风图（Tornado Chart）展示各参数对 NPV 的影响排序：
- 对每个参数 ±20% 变动，计算 NPV 变化范围
- 按影响大小降序排列，形成龙卷风形状
- 识别**高杠杆点**：小投入改变大局的关键参数

**2026前沿 — 推理成本对 AI 估值的影响**：
推理成本是 AI SaaS 估值的核心变量。DeepSeek 等开源模型将推理成本降低 90%+，
直接提升毛利率和估值。敏感性分析帮助量化这一影响。


In [ ]:
# TODO 5：敏感性分析 + 龙卷风图
# 提示：1) 定义 calc_npv(arpu, inference_ratio, data_ratio, growth_mult, opex_mult, dr) 辅助函数
#          margin = 1 - inference_ratio - data_ratio
#          基准: inference_ratio=0.30, data_ratio=0.05, margin=0.65
#       2) 对5个参数各±20%: ARPU/Inference Cost/Growth/OpEx/Discount Rate
#          ARPU: arpu=24*(1+d)
#          Inference Cost: inference_ratio=0.30*(1+d)  (注意: 推理成本下降->毛利率上升->NPV上升)
#          Growth: growth_mult=1.0+d
#          OpEx: opex_mult=1.0+d
#          Discount Rate: dr=0.15*(1+d)
#       3) 计算 each 的 NPV_high 和 NPV_low, 按影响范围降序排列
#       4) 画水平条形图(龙卷风图)
# 要求：打印敏感性排序表 + 绘制龙卷风图

# ===== 你的代码 =====
sensitivity_df = None  # 敏感性分析结果DataFrame

raise NotImplementedError


## 5. 天道推演 × 投资评估（2026前沿）

> 与项目 CLAUDE.md「天道推演系统」同构。

天道推演是一种元认知沙盘推演能力——以天神视角俯视局势，构建无限可能的沙盘，
模拟不同决策路径下的未来走向。应用于投资评估：

| 天道推演能力 | 投资评估对应 | 实现方式 |
|-------------|------------|---------|
| 局势感知 | 市场环境建模 | 场景定义 |
| 因果链追踪 | 价值驱动因素分析 | 敏感性分析 |
| 沙盘模拟（3层） | 多路径推演 | Bull / Base / Bear |
| 概率评估 | 估值概率分布 | 蒙特卡洛模拟 |
| 最优路径推荐 | 投资决策 | NPV / IRR / PI |

**三路径推演**：Bull（乐观）/ Base（基准）/ Bear（悲观），每路径推演 3 层（immediate / near / far）。


In [ ]:
# TODO 6：天道推演多路径场景分析
# 提示：1) 定义3个场景: Bull/Base/Bear, 各含5个参数(arpu/inference_ratio/growth_mult/opex_mult/dr)
#          data_ratio固定=0.05, margin = 1 - inference_ratio - data_ratio
#          Bull: arpu=28, inference_ratio=0.23, growth_mult=1.3, opex_mult=0.9, dr=0.12
#          Base: arpu=24, inference_ratio=0.30, growth_mult=1.0, opex_mult=1.0, dr=0.15
#          Bear: arpu=20, inference_ratio=0.40, growth_mult=0.7, opex_mult=1.2, dr=0.20
#       2) 每场景计算NPV (用calc_npv函数, 传data_ratio=0.05)
#       3) 对每场景做3层推演: immediate(Y1-2) / near(Y3-4) / far(Y5)
#       4) 打印场景对比表 + 天道推演风险预警
# 要求：构建3场景对比, 每场景含3层推演分析

# ===== 你的代码 =====
scenario_results = None  # 3场景结果

raise NotImplementedError


## 6. 反思与前沿

### 反思问题
1. MarketingAgent Pro 的 NPV 是多少？IRR 是否高于折现率？投资可行吗？
2. 蒙特卡洛模拟的 P(NPV>0) 是多少？5% 和 95% 分位差距说明了什么？
3. 敏感性分析中哪个参数对 NPV 影响最大？推理成本（通过毛利率）排第几？
4. 天道推演的三场景中，Bear case 的 NPV 是多少？风险预警是什么？
5. 如果 DeepSeek 将推理成本降低 90%，毛利率提升后 NPV 如何变化？

### 2026前沿：贝叶斯估值（Bayesian Valuation）
传统 DCF 给出点估计 NPV，蒙特卡洛给出频率派分布。**贝叶斯估值**用 PyMC 构建参数的
后验分布，结合先验信息和观测数据，给出更稳健的估值后验分布。

### Day 1-5 整合（技能4收官）
| Day | 能力 | Day 5 整合角色 |
|-----|------|--------------|
| Day 1 | AI 商业模式类型学 | 画布的客户细分 + 价值主张 |
| Day 2 | AI 定价策略 | 画布的收入流（outcome-based） |
| Day 3 | Agent 经济学 | 画布的成本结构（推理成本） |
| Day 4 | 平台生态战略 | 画布的核心伙伴 + 渠道 |
| Day 5 | 商业模式画布 + 投资评估 | **整合为完整投资评估** |
